<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/week3d_violin_rocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════
#  ЯЧЕЙКА 0. Подготовка данных (из week2b_read_csv.ipynb)
#  Запускать первой в каждом ноутбуке задания 3
# ═══════════════════════════════════════════════════════

# --- Параметры (изменять здесь) ----------------------
RADIUS_KM     = 300   # радиус соседства гор (для week3a)
TOP_N_ROCKS   = 10    # сколько топ-пород использовать
TOP_N_COMPLEX = 20    # сколько самых «сложных» гор брать
# -----------------------------------------------------

import os, pandas as pd, numpy as np
from itertools import combinations

# 1. Клонируем репозиторий (если ещё нет)
repo = "python-ai-AnastasiaKalyashova"
repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git
if os.getcwd() != repo_path:
    %cd {repo_path}

# 2. Читаем CSV
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break
df = pd.read_csv(file_path)

# 3. Переименование столбцов
if "mountainLabel" in df.columns:
    df = df.rename(columns={
        "mountain":          "URL",
        "mountainLabel":     "mountain",
        "rockMaterialLabel": "rockMaterial",
        "elevationMeters":   "elevation",
    })

# 4. Нормализуем породы
df["rockMaterial"] = df["rockMaterial"].str.lower().str.strip()

# 🔧 ИСПРАВЛЕНИЕ: заменяем "lutite" на "пелит"
df["rockMaterial"] = df["rockMaterial"].replace("lutite", "пелит")

# 5. Парсим координаты
coords = df["coordinates"].str.extract(r'Point\(([^\s]+)\s+([^\s]+)\)')
df["lon"] = pd.to_numeric(coords[0], errors="coerce")
df["lat"] = pd.to_numeric(coords[1], errors="coerce")

# 6. df_unique — по одной строке на гору
df_unique = (
    df.groupby("URL")
    .agg(
        mountain   = ("mountain",     "first"),
        lon        = ("lon",          "first"),
        lat        = ("lat",          "first"),
        elevation  = ("elevation",    "first"),
        rock_count = ("rockMaterial", "nunique"),
        rocks      = ("rockMaterial", lambda x: list(x.unique())),
    )
    .reset_index()
)

# 7. df_clean — только физически возможные высоты
df_clean = df_unique[
    (df_unique.elevation >= 0) &
    (df_unique.elevation <= 8849)
].copy()

# 8. Топ пород по частоте (по df_clean)
top_rocks = (
    df[df["URL"].isin(df_clean["URL"])]
    ["rockMaterial"].value_counts()
    .head(TOP_N_ROCKS).index.tolist()
)

# 9. Co-occurrence матрица пород
pairs = []
for rocks in df_clean["rocks"]:
    clean = [r for r in rocks if r in top_rocks]
    pairs += list(combinations(sorted(set(clean)), 2))
cooc = (pd.DataFrame(pairs, columns=["r1", "r2"])
        .value_counts()
        .reset_index(name="count"))

print(f"✅ Длинный формат:    {len(df)} строк")
print(f"✅ Уникальных гор:    {len(df_unique)}")
print(f"✅ df_clean:          {len(df_clean)} гор (0–8849 м)")
print(f"✅ Топ-{TOP_N_ROCKS} пород:    {top_rocks}")
print(f"✅ Пар co-occurrence: {len(cooc)}")

/content/python-ai-AnastasiaKalyashova
✅ Длинный формат:    4431 строк
✅ Уникальных гор:    2915
✅ df_clean:          2914 гор (0–8849 м)
✅ Топ-10 пород:    ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
✅ Пар co-occurrence: 15


In [30]:
# ═══════════════════════════════════════════════════════
# week3d_violin_rocks_alt.ipynb — Альтернативная визуализация
# "Эволюция пород с высотой": легенда между графиками
# ═══════════════════════════════════════════════════════

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
from scipy import stats

print("📊 Альтернативная визуализация: Эволюция пород с высотой")
print(f"   Топ-{TOP_N_ROCKS} пород: {top_rocks}")

# 1. Подготавливаем данные
df_violin = df[
    df["rockMaterial"].isin(top_rocks) &
    df["URL"].isin(df_clean["URL"])
].copy()

rock_count_dict = df_clean.set_index("URL")["rock_count"].to_dict()
df_violin["rock_count"] = df_violin["URL"].map(rock_count_dict)
df_violin["complexity"] = df_violin["rock_count"].apply(
    lambda x: "single" if x == 1 else "multi"
)

# 2. Вычисляем медианную высоту для каждой породы (сортируем по ней)
median_heights = df_violin.groupby("rockMaterial")["elevation"].median().sort_values()
sorted_rocks = median_heights.index.tolist()
print(f"   Породы от низких к высоким: {sorted_rocks}")

# 3. Создаём фигуру с ТРЕМЯ подграфиками (левый график, легенда, правый график)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("📊 Распределение высот",
                    "",  # Заголовок для легенды (пустой)
                    "📈 Медианные высоты"),
    specs=[
        [{"type": "scatter"}, {"type": "scatter"}, {"type": "bar"}]
    ],
    column_widths=[0.4, 0.15, 0.45],  # Увеличил легенду до 15%, правый график до 45%
    horizontal_spacing=0.08  # Увеличил расстояние между графиками
)

# ============== ЛЕВЫЙ ГРАФИК: Violin + точки ==============
for i, rock in enumerate(sorted_rocks):
    rock_data = df_violin[df_violin["rockMaterial"] == rock]

    # Разделяем по сложности
    single_data = rock_data[rock_data["complexity"] == "single"]
    multi_data = rock_data[rock_data["complexity"] == "multi"]

    # Основная скрипка (все данные)
    fig.add_trace(go.Violin(
        y=rock_data["elevation"],
        x=[rock] * len(rock_data),
        legendgroup=rock,
        scalegroup=rock,
        side='both',
        line_color='lightgray',
        fillcolor='rgba(100, 100, 100, 0.2)',
        width=0.8,
        showlegend=False,
        name=rock,
        hovertemplate='<b>%{text}</b><br>Высота: %{y:.0f} м<extra></extra>',
        text=rock_data["mountain"].tolist(),
    ), row=1, col=1)

    # Однопородные (синие кружочки)
    if len(single_data) > 0:
        fig.add_trace(go.Scatter(
            y=single_data["elevation"],
            x=[rock] * len(single_data),
            mode='markers',
            marker=dict(
                size=6,
                color='#1f77b4',
                opacity=0.6,
                symbol='circle',
                line=dict(width=0.5, color='white')
            ),
            name=f"{rock}_single",
            legendgroup=f"{rock}_single",
            showlegend=False,
            hovertemplate='<b>%{text}</b><br>🏔️ Высота: %{y:.0f} м<br>📚 1 порода<extra></extra>',
            text=single_data["mountain"].tolist(),
        ), row=1, col=1)

    # Многопородные (оранжевые ромбики)
    if len(multi_data) > 0:
        fig.add_trace(go.Scatter(
            y=multi_data["elevation"],
            x=[rock] * len(multi_data),
            mode='markers',
            marker=dict(
                size=6,
                color='#ff7f0e',
                opacity=0.6,
                symbol='diamond',
                line=dict(width=0.5, color='white')
            ),
            name=f"{rock}_multi",
            legendgroup=f"{rock}_multi",
            showlegend=False,
            hovertemplate='<b>%{text}</b><br>🗻 Высота: %{y:.0f} м<br>📚 ≥2 пород<extra></extra>',
            text=multi_data["mountain"].tolist(),
        ), row=1, col=1)

# ============== ЦЕНТРАЛЬНЫЙ ГРАФИК: ЛЕГЕНДА ==============
# Создаём текстовую легенду в центре
legend_text = f"""
<b>📖 УСЛОВНЫЕ ОБОЗНАЧЕНИЯ</b><br>
<br>
🔵 <span style="color:#1f77b4">●</span> <b>Однопородные горы</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;(только 1 тип породы)<br>
<br>
🟠 <span style="color:#ff7f0e">◆</span> <b>Многопородные горы</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;(≥2 типов пород)<br>
<br>
🔘 <span style="color:gray">▬</span> <b>Violin plot</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;(распределение высот)<br>
<br>
📊 <span style="color:steelblue">■</span> <b>Медианная высота</b><br>
<br>
📏 <span style="color:darkred">⎯</span> <b>IQR</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;(25-й и 75-й перцентили)<br>
<br>
<hr width="80%">
<b>📊 СТАТИСТИКА</b><br>
<br>
• Всего пород: <b>{len(sorted_rocks)}</b><br>
• Всего записей: <b>{len(df_violin)}</b><br>
• Уникальных гор: <b>{df_violin["URL"].nunique()}</b><br>
<br>
<hr width="80%">
<b>🎯 КЛЮЧЕВЫЕ ВЫВОДЫ</b><br>
<br>
🏔️ <b>Низкие горы</b><br>
&nbsp;&nbsp;&nbsp;→ {sorted_rocks[0]}<br>
&nbsp;&nbsp;&nbsp;→ {sorted_rocks[1] if len(sorted_rocks) > 1 else ""}<br>
<br>
🗻 <b>Высокие горы</b><br>
&nbsp;&nbsp;&nbsp;→ {sorted_rocks[-1]}<br>
&nbsp;&nbsp;&nbsp;→ {sorted_rocks[-2] if len(sorted_rocks) > 1 else ""}<br>
"""

# Добавляем легенду как текстовую аннотацию в центре
fig.add_annotation(
    x=0.5,
    y=0.5,
    xref="paper",  # Относительно всей фигуры
    yref="paper",
    text=legend_text,
    showarrow=False,
    font=dict(size=11),
    align="left",
    valign="middle",
    bordercolor="black",
    borderwidth=2,
    borderpad=15,
    bgcolor="rgba(250, 250, 250, 0.95)"
)

# Скрываем оси для центрального графика (легенды)
fig.update_xaxes(visible=False, row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)

# ============== ПРАВЫЙ ГРАФИК: Барплот с медианами ==============
# Вычисляем статистики
stats_data = []
for rock in sorted_rocks:
    rock_data = df_violin[df_violin["rockMaterial"] == rock]["elevation"]
    stats_data.append({
        "rock": rock,
        "median": rock_data.median(),
        "q1": rock_data.quantile(0.25),
        "q3": rock_data.quantile(0.75),
        "min": rock_data.min(),
        "max": rock_data.max(),
        "count": len(rock_data)
    })

stats_df = pd.DataFrame(stats_data)

# Бары для медиан
fig.add_trace(go.Bar(
    y=stats_df["rock"],
    x=stats_df["median"],
    orientation='h',
    marker_color='steelblue',
    marker_line_color='darkblue',
    marker_line_width=1,
    opacity=0.7,
    name='Медианная высота',
    hovertemplate='<b>%{y}</b><br>Медиана: %{x:.0f} м<br>Гор: %{customdata}<extra></extra>',
    customdata=stats_df["count"],
), row=1, col=3)

# Добавляем error bars (IQR)
fig.add_trace(go.Scatter(
    y=stats_df["rock"],
    x=stats_df["median"],
    error_x=dict(
        type='data',
        symmetric=False,
        array=stats_df["q3"] - stats_df["median"],
        arrayminus=stats_df["median"] - stats_df["q1"],
        color='rgba(0,0,0,0.5)',
        thickness=1.5,
        width=10
    ),
    mode='markers',
    marker=dict(size=8, color='darkred'),
    name='IQR (25-й и 75-й перцентили)',
    showlegend=False,  # Легенду уже добавили отдельно
    hovertemplate='<b>%{y}</b><br>Медиана: %{x:.0f} м<br>Q1: %{customdata[0]:.0f} м<br>Q3: %{customdata[1]:.0f} м<extra></extra>',
    customdata=stats_df[["q1", "q3"]].values,
), row=1, col=3)

# ============== НАСТРОЙКИ ЛЕВОГО ГРАФИКА ==============
fig.update_xaxes(
    title_text="<b>Тип горной породы</b>",
    tickangle=-45,
    tickfont=dict(size=9),
    row=1, col=1
)

fig.update_yaxes(
    title_text="<b>Высота (метры)</b>",
    gridcolor='lightgray',
    gridwidth=0.5,
    row=1, col=1
)

# ============== НАСТРОЙКИ ПРАВОГО ГРАФИКА ==============
fig.update_xaxes(
    title_text="<b>Высота (метры)</b>",
    gridcolor='lightgray',
    gridwidth=0.5,
    range=[0, stats_df["median"].max() * 1.1],  # Добавляем небольшой отступ справа
    row=1, col=3
)

fig.update_yaxes(
    title_text="<b>Тип горной породы</b>",
    tickfont=dict(size=9),
    row=1, col=3
)

# ============== ОБЩИЕ НАСТРОЙКИ ==============
fig.update_layout(
    title=dict(
        text="<b>🗻 Эволюция горных пород с высотой</b><br>" +
             "<sup>Как меняется состав гор в зависимости от высоты? (породы отсортированы по медианной высоте)</sup>",
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    height=800,
    width=1800,  # Увеличил общую ширину
    hovermode='closest',
    showlegend=False,  # Отключаем авто-легенду, используем кастомную
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=50, r=50, t=100, b=50)  # Добавил отступы
)

# Добавляем горизонтальную линию 5000 м на левом графике
fig.add_hline(y=5000, line_dash="dash", line_color="red", opacity=0.5,
              annotation_text="5000 м", annotation_position="top right",
              row=1, col=1)

# Добавляем сетку для правого графика
fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor='lightgray', row=1, col=3)

# Удаляем заголовок для центральной колонки
fig.layout.annotations[1].text = ""  # Убираем заголовок ""

fig.show()

# ============== ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ ==============
print("\n" + "="*70)
print("📊 АНАЛИЗ: Эволюция пород с высотой")
print("="*70)

# 1. Корреляция между высотой и типом породы
print("\n📈 Породы, характерные для разных высотных зон:")

# Низкие горы (<1000 м)
low_rocks = df_violin[df_violin["elevation"] < 1000]["rockMaterial"].value_counts().head(3)
print(f"\n   🟢 НИЗКИЕ ГОРЫ (<1000 м):")
for rock, count in low_rocks.items():
    print(f"      • {rock}: {count} записей")

# Средние горы (1000-3000 м)
mid_rocks = df_violin[(df_violin["elevation"] >= 1000) & (df_violin["elevation"] < 3000)]["rockMaterial"].value_counts().head(3)
print(f"\n   🟡 СРЕДНИЕ ГОРЫ (1000-3000 м):")
for rock, count in mid_rocks.items():
    print(f"      • {rock}: {count} записей")

# Высокие горы (≥3000 м)
high_rocks = df_violin[df_violin["elevation"] >= 3000]["rockMaterial"].value_counts().head(3)
print(f"\n   🔴 ВЫСОКИЕ ГОРЫ (≥3000 м):")
for rock, count in high_rocks.items():
    print(f"      • {rock}: {count} записей")

# 2. Статистика по сложности
print("\n" + "="*70)
print("🧬 ВЛИЯНИЕ СЛОЖНОСТИ (однопородные vs многопородные):")
print("="*70)

complexity_stats = df_violin.groupby("complexity")["elevation"].agg(["mean", "median", "std", "count"])
print(complexity_stats.round(1).to_string())

# 3. T-тест
single_heights = df_violin[df_violin["complexity"] == "single"]["elevation"]
multi_heights = df_violin[df_violin["complexity"] == "multi"]["elevation"]
if len(single_heights) > 0 and len(multi_heights) > 0:
    t_stat, p_value = stats.ttest_ind(single_heights, multi_heights)
    print(f"\n📊 Статистический тест (t-test):")
    print(f"   t-статистика: {t_stat:.3f}")
    print(f"   p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("   ✅ Разница статистически значима (p < 0.05)")
        if single_heights.mean() > multi_heights.mean():
            print("   → Однопородные горы в среднем ВЫШЕ многопородных")
        else:
            print("   → Многопородные горы в среднем ВЫШЕ однопородных")
    else:
        print("   ❌ Разница статистически не значима")

# 4. Тренд
print("\n" + "="*70)
print("📈 ТРЕНД: Рост медианной высоты")
print("="*70)

# Линейная регрессия
x_pos = np.arange(len(sorted_rocks))
y_med = stats_df["median"].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x_pos, y_med)

print(f"   Коэффициент наклона: {slope:.1f} м/порода")
print(f"   R²: {r_value**2:.3f}")
print(f"   p-value: {p_value:.4f}")

if slope > 0 and p_value < 0.05:
    print("   ✅ Породы имеют ТЕНДЕНЦИЮ к увеличению высоты!")
    print(f"   → Каждая следующая порода (в отсортированном списке) в среднем выше на {slope:.0f} м")
elif slope < 0 and p_value < 0.05:
    print("   📉 Породы имеют тенденцию к снижению высоты")
else:
    print("   ➡️ Явной тенденции не обнаружено")

# 5. Вывод
print("\n" + "="*70)
print("💎 ГЛАВНЫЙ ВЫВОД:")
print("="*70)
if slope > 50 and p_value < 0.05:
    print(f"   🏔️ Породы ЭВОЛЮЦИОНИРУЮТ с высотой!")
    print(f"   Низкие горы → {sorted_rocks[0]}")
    print(f"   Высокие горы → {sorted_rocks[-1]}")
    print(f"   Разница в медианной высоте: {y_med[-1] - y_med[0]:.0f} м")
else:
    print(f"   📊 Самые низкие горы: {sorted_rocks[0]} ({y_med[0]:.0f} м)")
    print(f"   📊 Самые высокие горы: {sorted_rocks[-1]} ({y_med[-1]:.0f} м)")

📊 Альтернативная визуализация: Эволюция пород с высотой
   Топ-10 пород: ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
   Породы от низких к высоким: ['мергель', 'песчаник', 'конгломерат', 'доломит', 'известняк', 'базальт', 'пелит', 'осадочная горная порода', 'андезит', 'гранит']



📊 АНАЛИЗ: Эволюция пород с высотой

📈 Породы, характерные для разных высотных зон:

   🟢 НИЗКИЕ ГОРЫ (<1000 м):
      • известняк: 367 записей
      • песчаник: 296 записей
      • мергель: 222 записей

   🟡 СРЕДНИЕ ГОРЫ (1000-3000 м):
      • известняк: 449 записей
      • песчаник: 217 записей
      • гранит: 217 записей

   🔴 ВЫСОКИЕ ГОРЫ (≥3000 м):
      • гранит: 96 записей
      • андезит: 63 записей
      • известняк: 22 записей

🧬 ВЛИЯНИЕ СЛОЖНОСТИ (однопородные vs многопородные):
              mean  median     std  count
complexity                               
multi       1190.1   864.5   820.0   1626
single      1891.9  1745.0  1174.6   1587

📊 Статистический тест (t-test):
   t-статистика: 19.675
   p-value: 0.0000
   ✅ Разница статистически значима (p < 0.05)
   → Однопородные горы в среднем ВЫШЕ многопородных

📈 ТРЕНД: Рост медианной высоты
   Коэффициент наклона: 219.2 м/порода
   R²: 0.931
   p-value: 0.0000
   ✅ Породы имеют ТЕНДЕНЦИЮ к увеличению высоты!
   → Каждая

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# week3d_violin_rocks.ipynb — Голос каждой породы
# Структурная визуализация: как одна и та же гора «переходит» между породами
# ═══════════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
from scipy import stats

print("📊 Структурная визуализация: как породы связываются внутри гор")
print(f"   Топ-{TOP_N_ROCKS} пород: {top_rocks}")

# ==============================================================================
# 1. ПОДГОТОВКА ДАННЫХ
# ==============================================================================

# Фильтруем данные по топ-породам
df_violin = df[
    df["rockMaterial"].isin(top_rocks) &
    df["URL"].isin(df_clean["URL"])
].copy()

# ==============================================================================
# 2. ОПТИМИЗАЦИЯ ПОРЯДКА СКРИПОК (уменьшаем дальние связи)
# ==============================================================================

# Начальный порядок по медианной высоте
median_heights = df_violin.groupby("rockMaterial")["elevation"].median().sort_values()
initial_order = median_heights.index.tolist()

# Строим пары пород, которые встречаются вместе
cooc_pairs = []
for rocks in df_clean["rocks"]:
    clean_rocks = [r for r in rocks if r in top_rocks]
    for i, a in enumerate(clean_rocks):
        for b in clean_rocks[i+1:]:
            cooc_pairs.append((a, b))

pairs = list(set(cooc_pairs))

def total_span(order, pairs):
    pos = {rock: i for i, rock in enumerate(order)}
    return sum(abs(pos[a] - pos[b]) for a, b in pairs)

# Локальная оптимизация порядка
order = initial_order.copy()
improved = True
while improved:
    improved = False
    for i in range(len(order) - 1):
        cand = order.copy()
        cand[i], cand[i+1] = cand[i+1], cand[i]
        if total_span(cand, pairs) < total_span(order, pairs):
            order = cand
            improved = True

sorted_rocks = order
print(f"\n📐 Порядок скрипок (оптимизирован под связи):")
for i, rock in enumerate(sorted_rocks):
    print(f"   {i}. {rock}")

# ==============================================================================
# 3. ЦВЕТА ДЛЯ ПОРОД
# ==============================================================================

ROCK_COLORS = {
    rock: color
    for rock, color in zip(
        sorted_rocks,
        px.colors.qualitative.Set3[:len(sorted_rocks)]
    )
}

x_pos = {rock: i for i, rock in enumerate(sorted_rocks)}

# ==============================================================================
# 4. СОЗДАНИЕ ГРАФИКА
# ==============================================================================

fig = go.Figure()

# ---- 4.1. Скрипки ----
for rock in sorted_rocks:
    rock_data = df_violin[df_violin["rockMaterial"] == rock]

    fig.add_trace(go.Violin(
        y=rock_data["elevation"],
        x=[x_pos[rock]] * len(rock_data),
        legendgroup=rock,
        scalegroup=rock,
        side='both',
        line_color='lightgray',
        fillcolor='rgba(150, 150, 150, 0.15)',
        width=0.7,
        showlegend=False,
        name=rock,
        hovertemplate='<b>%{text}</b><br>Высота: %{y:.0f} м<extra></extra>',
        text=rock_data["mountain"].tolist(),
    ))

# ---- 4.2. Цветные кружки под скрипками (y = -300) ----
fig.add_trace(go.Scatter(
    x=[x_pos[r] for r in sorted_rocks],
    y=[-300] * len(sorted_rocks),
    mode="markers",
    marker=dict(
        size=22,
        color=[ROCK_COLORS[r] for r in sorted_rocks],
        line=dict(color="black", width=1.5)
    ),
    showlegend=False,
    hoverinfo="skip"
))

# ---- 4.3. Текстовые подписи пород (y = -450, дальше от кружков) ----
fig.add_trace(go.Scatter(
    x=[x_pos[r] for r in sorted_rocks],
    y=[-480] * len(sorted_rocks),  # Увеличил расстояние с -430 до -480
    mode="text",
    text=[r for r in sorted_rocks],
    textfont=dict(size=10, color="black", family="Arial"),
    showlegend=False,
    hoverinfo="skip"
))

# ---- 4.4. Точки на скрипках ----
for rock in sorted_rocks:
    rock_data = df_violin[df_violin["rockMaterial"] == rock]

    fig.add_trace(go.Scatter(
        x=[x_pos[rock]] * len(rock_data),
        y=rock_data["elevation"],
        mode='markers',
        marker=dict(
            size=5,
            color=ROCK_COLORS[rock],
            opacity=0.7,
            symbol='circle',
            line=dict(width=0.5, color='white')
        ),
        showlegend=False,
        hovertemplate='<b>%{text}</b><br>🏔️ Высота: %{y:.0f} м<br>📚 Порода: ' + rock + '<extra></extra>',
        text=rock_data["mountain"].tolist(),
    ))

# ==============================================================================
# 5. СВЯЗИ МЕЖДУ ПОРОДАМИ
# ==============================================================================

def arc_points(x0, x1, y0, height=200):
    xs = np.linspace(x0, x1, 50)
    xm = (x0 + x1) / 2
    ys = y0 + height * (1 - ((xs - xm) / ((x1 - x0) / 2 + 1e-9))**2)
    return xs, ys

links_data = []
for _, row in df_clean.iterrows():
    rocks_in_mountain = [r for r in row["rocks"] if r in set(sorted_rocks)]
    if len(rocks_in_mountain) < 2:
        continue
    for i, from_rock in enumerate(rocks_in_mountain):
        for to_rock in rocks_in_mountain[i+1:]:
            links_data.append({
                "mountain": row["mountain"],
                "elevation": row["elevation"],
                "from_rock": from_rock,
                "to_rock": to_rock
            })

links_df = pd.DataFrame(links_data)
print(f"\n🔗 Всего связей между породами: {len(links_df)}")

unique_links = links_df.groupby(["from_rock", "to_rock"]).first().reset_index()

for _, link in unique_links.iterrows():
    from_rock = link["from_rock"]
    to_rock = link["to_rock"]

    x0 = x_pos[from_rock]
    x1 = x_pos[to_rock]
    y = link["elevation"]
    is_adjacent = abs(x0 - x1) == 1

    if is_adjacent:
        fig.add_trace(go.Scatter(
            x=[x0, x1],
            y=[y, y],
            mode="lines+markers",
            line=dict(color=ROCK_COLORS[to_rock], width=1.5),
            marker=dict(size=[0, 6], color=[ROCK_COLORS[from_rock], ROCK_COLORS[to_rock]]),
            showlegend=False,
            hoverinfo="skip"
        ))
    else:
        height = 150 + 40 * abs(x1 - x0)
        xs, ys = arc_points(x0, x1, y, height=height)

        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode="lines",
            line=dict(color=ROCK_COLORS[to_rock], width=1), opacity=0.35,
            showlegend=False, hoverinfo="skip"
        ))

        fig.add_trace(go.Scatter(
            x=[x1], y=[y], mode="markers",
            marker=dict(size=6, color=ROCK_COLORS[to_rock], line=dict(color="black", width=0.5)),
            showlegend=False,
            hovertemplate=f"<b>{link['mountain']}</b><br>{to_rock}<br>{y:.0f} м<extra></extra>"
        ))

# ==============================================================================
# 6. ПОДПИСИ ВЫДАЮЩИХСЯ ВЕРШИН
# ==============================================================================

LABEL_MOUNTAINS = [
    "Джомолунгма",
    "К2",
    "Казбек",
    "Пик Ленина",
    "Эльбрус"
]

label_df = df_violin[df_violin["mountain"].isin(LABEL_MOUNTAINS)].copy()

if len(label_df) > 0:
    label_df = label_df.sort_values("elevation", ascending=False).drop_duplicates("mountain")
    label_df["x"] = label_df["rockMaterial"].map(x_pos)

    print(f"\n🏔️ Подписанные горы:")
    for _, row in label_df.iterrows():
        print(f"   • {row['mountain']}: {row['elevation']:.0f} м (порода: {row['rockMaterial']})")

    fig.add_trace(go.Scatter(
        x=label_df["x"],
        y=label_df["elevation"],
        mode="markers+text",
        text=label_df["mountain"],
        textposition="top center",
        textfont=dict(size=11, color="black", family="Arial Black"),
        marker=dict(
            size=14,
            color=label_df["rockMaterial"].map(ROCK_COLORS),
            line=dict(color="black", width=1.5),
            symbol="star"
        ),
        showlegend=False,
        hovertemplate="<b>%{text}</b><br>⭐ %{y:.0f} м<extra></extra>"
    ))
else:
    print(f"\n⚠️ ВНИМАНИЕ: Ни одна из гор {LABEL_MOUNTAINS} не найдена в df_violin!")

# ==============================================================================
# 7. НАСТРОЙКИ ВНЕШНЕГО ВИДА
# ==============================================================================

# Расширяем диапазон Y, чтобы поместить подписи
y_min = -550  # Увеличил с -500 до -550

fig.update_layout(
    title=dict(
        text="<b>🗻 Голос каждой породы: как горы связывают разные породы</b><br>" +
             "<sup>Скрипки показывают распределение высот; цветные точки — горы; линии и дуги — связи между породами внутри одной горы</sup>",
        x=0.4,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(
        title="<b>Тип горной породы</b>",
        tickfont=dict(size=11),
        tickmode="array",
        tickvals=list(range(len(sorted_rocks))),
        ticktext=[],  # Убираем стандартные подписи
        range=[-0.5, len(sorted_rocks) - 0.5]
    ),
    yaxis=dict(
        title="<b>Высота (метры)</b>",
        gridcolor='lightgray',
        gridwidth=0.5,
        range=[y_min, df_violin["elevation"].max() + 500]
    ),
    height=950,  # Увеличил высоту с 900 до 950
    width=1800,
    hovermode='closest',
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=450, t=100, b=100),  # Увеличил нижний отступ
    showlegend=False
)

fig.add_hline(y=5000, line_dash="dash", line_color="red", opacity=0.4,
              annotation_text="5000 м", annotation_position="top right")

# ==============================================================================
# 8. БЛОК «УСЛОВНЫЕ ОБОЗНАЧЕНИЯ»
# ==============================================================================

n_rocks = len(sorted_rocks)
n_records = len(df_violin)
n_mountains = df_violin["URL"].nunique()
n_links = len(unique_links)

labeled_found = [m for m in LABEL_MOUNTAINS if m in df_violin["mountain"].values]
labeled_str = ", ".join(labeled_found) if labeled_found else "не найдены"

legend_text = f"""
<b>📖 УСЛОВНЫЕ ОБОЗНАЧЕНИЯ</b><br>
<br>
<span style="color:gray">▬</span> <b>Серая скрипка</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;распределение высот породы<br>
<br>
<span style="color:black">●</span> <b>Цветная точка</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;гора, содержащая эту породу<br>
<br>
<span style="color:black">◆</span> <b>Цветной кружок под скрипкой</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;цвет, закреплённый за породой<br>
<br>
<span style="color:gray">━━</span> <b>Горизонтальный отрезок</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;связь с соседней породой<br>
<br>
<span style="color:gray">◡</span> <b>Дуга</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;связь с далёкой породой<br>
<br>
⭐ <b>Звезда с подписью</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;выдающаяся вершина<br>
<br>
<b>📊 СТАТИСТИКА</b><br>
<br>
• Пород: <b>{n_rocks}</b><br>
• Записей: <b>{n_records}</b><br>
• Гор: <b>{n_mountains}</b><br>
• Связей: <b>{n_links}</b><br>
<br>
<b>🏔️ ПОДПИСАННЫЕ ВЕРШИНЫ</b><br>
<br>
{labeled_str}<br>
<br>
<b>🎯 КЛЮЧЕВОЙ ВОПРОС</b><br>
<br>
Как одна и та же гора<br>
«переходит» между разными<br>
породами по мере роста?
"""

fig.add_annotation(
    x=1.3,
    y=0.5,
    xref="paper",
    yref="paper",
    text=legend_text,
    showarrow=False,
    font=dict(size=11),
    align="left",
    valign="middle",
    bordercolor="black",
    borderwidth=1.5,
    borderpad=12,
    bgcolor="rgba(255, 255, 255, 0.95)"
)

fig.add_annotation(
    x=1.08,
    y=0.5,
    xref="paper",
    yref="paper",
    text="→",
    showarrow=False,
    font=dict(size=20, color="gray"),
    textangle=0
)

fig.show()

# ==============================================================================
# 9. ДИАГНОСТИКА
# ==============================================================================

print("\n" + "="*70)
print("🔍 ДИАГНОСТИКА: какие горы из списка попали в df_violin?")
print("="*70)

for mountain in LABEL_MOUNTAINS:
    in_df = mountain in df_violin["mountain"].values
    if in_df:
        row = df_violin[df_violin["mountain"] == mountain].iloc[0]
        print(f"   ✅ {mountain}: {row['elevation']:.0f} м, порода: {row['rockMaterial']}")
    else:
        in_clean = mountain in df_clean["mountain"].values
        if in_clean:
            row_clean = df_clean[df_clean["mountain"] == mountain].iloc[0]
            print(f"   ⚠️ {mountain}: есть в df_clean, но её породы нет в топ-{TOP_N_ROCKS}")
            print(f"      Породы этой горы: {row_clean['rocks']}")
        else:
            print(f"   ❌ {mountain}: нет в данных")

print("\n" + "="*70)
print("💎 ГЛАВНЫЙ ВЫВОД:")
print("="*70)
print("   График показывает, как горы «переходят» из одних пород в другие.")

📊 Структурная визуализация: как породы связываются внутри гор
   Топ-10 пород: ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']

📐 Порядок скрипок (оптимизирован под связи):
   0. мергель
   1. песчаник
   2. конгломерат
   3. пелит
   4. известняк
   5. доломит
   6. базальт
   7. андезит
   8. осадочная горная порода
   9. гранит

🔗 Всего связей между породами: 981

🏔️ Подписанные горы:
   • Эльбрус: 5642 м (порода: гранит)



🔍 ДИАГНОСТИКА: какие горы из списка попали в df_violin?
   ⚠️ Джомолунгма: есть в df_clean, но её породы нет в топ-10
      Породы этой горы: ['горная порода', 'лёд']
   ❌ К2: нет в данных
   ⚠️ Казбек: есть в df_clean, но её породы нет в топ-10
      Породы этой горы: ['трахит']
   ❌ Пик Ленина: нет в данных
   ✅ Эльбрус: 5642 м, порода: гранит

💎 ГЛАВНЫЙ ВЫВОД:
   График показывает, как горы «переходят» из одних пород в другие.
